In [4]:
import os
from dotenv import load_dotenv
from pathlib import Path

load_dotenv()

DATA_ROOT = Path(os.environ["DATA_ROOT"])
HF_HOME = DATA_ROOT / ".hf_cache"
os.environ["HF_HOME"] = str(HF_HOME)

import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer
from datasets import load_dataset

## 데이터셋 확인

In [2]:
dataset = load_dataset("ingyoun/patent-clean-text")
dataset

README.md:   0%|          | 0.00/760 [00:00<?, ?B/s]

c:\workspace\patent_disc\.venv\Lib\site-packages\huggingface_hub\file_download.py:139: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\workspace\patent_disc\.hf_cache\hub\datasets--ingyoun--patent-clean-text. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


data/train-00000-of-00002.parquet:   0%|          | 0.00/134M [00:00<?, ?B/s]

data/train-00001-of-00002.parquet:   0%|          | 0.00/134M [00:00<?, ?B/s]

data/validation-00000-of-00001.parquet:   0%|          | 0.00/15.0M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/15.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/201895 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/11216 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/11217 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['document_id', 'invention_title', 'abstract', 'claims', 'ipc_main', 'mno', 'label_ids', 'lno'],
        num_rows: 201895
    })
    validation: Dataset({
        features: ['document_id', 'invention_title', 'abstract', 'claims', 'ipc_main', 'mno', 'label_ids', 'lno'],
        num_rows: 11216
    })
    test: Dataset({
        features: ['document_id', 'invention_title', 'abstract', 'claims', 'ipc_main', 'mno', 'label_ids', 'lno'],
        num_rows: 11217
    })
})

In [5]:
model_name = "monologg/kobert"
REV = "38279184ba645e8c94d709fbe92eb5bcb47312c1"
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True, revision=REV)

In [10]:
train_ds = dataset["train"]
val_ds = dataset["validation"]
test_ds = dataset["test"]

for k, v in train_ds[0].items():
    print(f"{k}: {v}")

document_id: jp2012213809b2
invention_title: 수력발전장치
abstract: (과제) 간단한 구성으로, 낙차공이 없는 수로로 용이하게 설치할 수 있고, 수로를 흐르는 물의 에너지를 유효하게 이용하여 발전할 수 있는 수력발전장치를 제공한다. (해결수단) 수로 12 안으로 유로를 막도록 설치되어 수류를 받아 회전하는 란나 20과, 란나 20으로 연동하고 발전을 실시하는 발전기 50을 가진다. 란나 20을 유지한 란나 유지 테두리 36과, 란나 20을 덮도록 설치되어 수로 12의 수류가 란나 20의 하부의 블레이드 42에 흐르도록 가이드함과 동시에, 상류측의 수위를 조절하는 수위 조절 커버 54를 구비한다. 수위 조절 커버 54는 수로 12의 거의 전체 폭에 걸쳐 위치하고, 수심 방향의 위치를 조정 가능하게 설치되어 있다. 수위 조절 커버 54는 란나 유지 테두리 36에 장착된 커버 구동용 모터 56에 의해, 란나 20과 동축으로 요동한다. (선택도) 도 1.
claims: 수로 안으로 유로를 막도록 설치되어 수류를 받아 회전하는 란나와, 이 란나에 의해 구동되고 발전을 실시하는 발전기와, 상기 란나를 유지한 란나 유지 테두리와, 상기 란나를 덮도록 설치되어 상기 수로의 수류가 상기 란나의 하부의 블레이드로 흘러들도록 가이드함과 동시에, 상류측의 수위를 조절하는 수위 조절 커버를 구비하고, 상기 수위 조절 커버는 상기 란나를 덮고 상기 란나에 따라서 요동함으로써 상기 수로의 수심 방향의 위치를 조절 가능하게 설치되고, 상기 수위 조절 커버보다, 상기 란나의 상류측의 상기 수로의 수위와 상기 란나로 유입되는 수량을 조정하고, 상기 란나를 회전시켜 발전을 실시하는 것을 특징으로 하는 수력발전장치.
ipc_main: F03B-007/00
mno: ['EF03']
label_ids: [61]
lno: ['EF']


In [ ]:
def build_inputs(ex):
    fields = [
        ("명칭", ex["invention_title"]),
        ("IPC",  ex["ipc_main"]),
        ("요약", ex["abstract"]),
        ("청구항", ex["claims"]),
    ]
    text = " ".join(f"{k}: {v}" for k, v in fields if v)   # 빈 필드 skip, 개행/들여쓰기 없음
    return {"document_id": ex["document_id"], "input": text, "label_ids": ex["label_ids"]}

In [ ]:
inputs = 